# Solvanity
A CUDA address lab for genuine wallet keypairs and seeded Solana accounts.

Choose a GPU runtime in Colab, then run the two cells below. This notebook does not allocate or purchase a runtime.

**Cloud-generated wallets are for testing.** Google VM memory, browser state and saved notebook outputs are not a key vault. For real funds, run locally on a trusted offline machine. Never enter an existing private key. Do not save or share outputs or widget state containing results.

Seeded accounts use `create_with_seed`. They are not wallet keypairs or PDAs; you still need the base signer to create an account. No RPC calls or transactions are made.

In [ ]:
# Install both search engines. No private credentials are used.
import os, subprocess, sys
from pathlib import Path
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
root = Path("/content/solvanity") if Path("/content").exists() else Path.cwd() / ".solvanity"
root.mkdir(parents=True, exist_ok=True)
print(f"Python {sys.version.split()[0]} ({sys.executable})")
def setup_run(command):
    # Capture both streams explicitly: Colab can hide inherited subprocess stderr.
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout, end="", flush=True)
    result.check_returncode()
for repo in ("solvanity",):
    dest = root / repo
    if not dest.exists():
        setup_run(["git", "clone", "--depth", "1", f"https://github.com/koboshchan/{repo}.git", str(dest)])
    elif not (dest / ".git").is_dir():
        raise RuntimeError(f"{dest} exists but is not a Git checkout. Choose an empty setup directory.")
    else:
        # Recover an earlier failed setup without discarding local edits.
        setup_run(["git", "-C", str(dest), "pull", "--ff-only"])
    setup_run(["git", "-C", str(dest), "rev-parse", "HEAD"])
setup_run([sys.executable, "-m", "pip", "--verbose", "install", "ipywidgets>=8.1,<9", str(root/"solvanity")])
# Select CuPy for the runtime's actual driver capability, not a guessed GPU name.
try:
    import re
    driver = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True).stdout
    match = re.search(r"CUDA Version:\s*(\d+)", driver)
    major = int(match.group(1)) if match else 0
    if major < 12:
        raise RuntimeError("CUDA 12+ driver required. CPU mode remains available.")
    package = "cupy-cuda13x" if major >= 13 else "cupy-cuda12x"
    # Conflicting CUDA-major CuPy wheels share the same Python namespace.
    from importlib.metadata import distributions
    installed = {d.metadata["Name"].lower() for d in distributions()}
    conflicts = sorted(installed & ({"cupy", "cupy-cuda11x", "cupy-cuda12x", "cupy-cuda13x"} - {package}))
    if conflicts:
        setup_run([sys.executable, "-m", "pip", "uninstall", "-y", *conflicts])
    setup_run([sys.executable, "-m", "pip", "--verbose", "install", package])
    print("GPU dependencies ready. Open the search interface below.")
except (FileNotFoundError, subprocess.CalledProcessError, RuntimeError) as exc:
    print(f"GPU setup unavailable ({type(exc).__name__}). CPU mode is available; choose a GPU runtime to use CUDA.")


In [ ]:
# Open the shared UI. Results stay hidden until you explicitly reveal/export them.
import sys
sys.path.insert(0, str(root / "solvanity" / "notebooks"))
from solvanity_ui import SolvanityUI
solvanity = SolvanityUI().display()


When finished, download only the results you need, use **Forget results**, clear every notebook output and delete the runtime. Forgetting is not secure memory zeroization. The probability meter is a rough statistical estimate, not a percentage-complete indicator or a guaranteed deadline.